# T2.4 - View Definitions
Owner: D (El Dib Yehea)
Creates the ML pipeline views in DBRepo via the REST API.

In [1]:
import requests
import os
from dotenv import load_dotenv
from requests.auth import HTTPBasicAuth

load_dotenv()

BASE_URL    = "https://test.dbrepo.tuwien.ac.at"
USERNAME    = os.getenv("DBREPO_USERNAME")
PASSWORD    = os.getenv("DBREPO_PASSWORD")
DATABASE_ID = "82c19b39-246c-4409-b25c-8baf3a158a70"
TABLE_ID    = "9e7a7b18-58d9-4053-864e-82232463c8f5"


In [2]:
response = requests.get(
    f"{BASE_URL}/api/v1/database/{DATABASE_ID}",
    auth=HTTPBasicAuth(USERNAME, PASSWORD),
    headers={"Accept": "application/json"}
)

if response.status_code == 200:
    db = response.json()
    print(f"Connection successful")
    print(f"Database name : {db.get('name')}")
    print(f"Database ID   : {db.get('id')}")
else:
    print(f"Connection failed with status {response.status_code}")
    print(response.text[:200])

Connection successful
Database name : uk-collision-severity-prediction-main
Database ID   : 82c19b39-246c-4409-b25c-8baf3a158a70


In [3]:
VIEWS = [
    {
        "name": "collision_ml_features",
        "purpose": "Clean feature table matching exactly the input expected by 01_load_data.py. Contains the 15 features and label used for ML training. Primary source for T2.6 API reimplementation.",
        "body": {
            "name": "collision_ml_features",
            "is_public": True,
            "is_schema_public": True,
            "query": {
                "datasource_ids": [TABLE_ID],
                "columns": [
                    {"id": "e6021fe6-85d7-4e87-97ff-ba9c2975b028"},  # speed_limit
                    {"id": "83181255-1865-42ce-bf2d-61418a7fa8ec"},  # light_conditions
                    {"id": "29806930-1846-4d20-8e44-0d5c2d8bcfbc"},  # weather_conditions
                    {"id": "ef71fe54-beb2-4c55-ab60-65945a70c473"},  # road_surface_conditions
                    {"id": "c324b3eb-8c0a-4608-869b-d5ee60ea224a"},  # road_type
                    {"id": "fd531c20-94c6-4f44-8ed1-e3327e8b87d0"},  # urban_or_rural_area
                    {"id": "204d03a7-c2cd-4cbe-ac0e-2e574684af34"},  # number_of_vehicles
                    {"id": "ba565c16-a7e7-4318-87c3-9a78cd4e2c10"},  # number_of_casualties
                    {"id": "35b1830a-42dc-4bad-b419-f6df2323d531"},  # day_of_week
                    {"id": "64d5a206-0b12-4c93-95e0-0a039fda7226"},  # junction_detail
                    {"id": "cd7a7e8b-b80f-4538-a61e-0c46c7bf1f6e"},  # junction_control
                    {"id": "4e2ba1fc-e613-43ac-a5cc-1ba134767e05"},  # pedestrian_crossing
                    {"id": "f0825a1d-e556-40a7-8111-ff3cc0d87a80"},  # first_road_class
                    {"id": "a8be2630-dff6-4919-8b07-1a75c250658e"},  # special_conditions_at_site
                    {"id": "5e508f5c-23a2-4b26-b1ae-e24026dccec9"},  # carriageway_hazards
                    {"id": "e2d2e08a-fcf5-4912-87cf-975732f9fdb1"},  # collision_severity
                ],
                "joins": [],
                "filters": [],
                "orders": []
            }
        }
    },
    {
        "name": "collision_severity_summary",
        "purpose": "Aggregated collision counts grouped by severity, road type, urban/rural area and speed limit. Used to verify class imbalance before SMOTE balancing in 02_preprocess.py.",
        "body": {
            "name": "collision_severity_summary",
            "is_public": True,
            "is_schema_public": True,
            "query": {
                "datasource_ids": [TABLE_ID],
                "columns": [
                    {"id": "e2d2e08a-fcf5-4912-87cf-975732f9fdb1"},  # collision_severity
                    {"id": "fd531c20-94c6-4f44-8ed1-e3327e8b87d0"},  # urban_or_rural_area
                    {"id": "c324b3eb-8c0a-4608-869b-d5ee60ea224a"},  # road_type
                    {"id": "e6021fe6-85d7-4e87-97ff-ba9c2975b028"},  # speed_limit
                    {"id": "ba565c16-a7e7-4318-87c3-9a78cd4e2c10"},  # number_of_casualties
                    {"id": "204d03a7-c2cd-4cbe-ac0e-2e574684af34"},  # number_of_vehicles
                ],
                "joins": [],
                "filters": [],
                "orders": []
            }
        }
    }
]

print(f"Defined {len(VIEWS)} views and ready to create")
for v in VIEWS:
    print(f"  - {v['name']}: {v['purpose'][:60]}...")

Defined 2 views and ready to create
  - collision_ml_features: Clean feature table matching exactly the input expected by 0...
  - collision_severity_summary: Aggregated collision counts grouped by severity, road type, ...


In [4]:
created_views = {}

for view in VIEWS:
    print(f"Creating view: {view['name']}")

    response = requests.post(
        f"{BASE_URL}/api/v1/database/{DATABASE_ID}/view",
        auth=HTTPBasicAuth(USERNAME, PASSWORD),
        headers={
            "Content-Type": "application/json",
            "Accept": "application/json"
        },
        json=view["body"]
    )

    print(f"  Status: {response.status_code}")

    if response.status_code in (200, 201):
        view_id = response.json().get("id")
        created_views[view["name"]] = view_id
        print(f"  Created successfully with ID: {view_id}")
    else:
        print(f"  Failed: {response.text[:300]}")

print(f"\nTotal views created: {len(created_views)} out of {len(VIEWS)}")

Creating view: collision_ml_features
  Status: 403
  Failed: {"status":"FORBIDDEN","message":"Failed to create view: not the database owner","code":"error.request.forbidden"}
Creating view: collision_severity_summary
  Status: 403
  Failed: {"status":"FORBIDDEN","message":"Failed to create view: not the database owner","code":"error.request.forbidden"}

Total views created: 0 out of 2


In [5]:
response = requests.get(
    f"{BASE_URL}/api/v1/database/{DATABASE_ID}/view",
    auth=HTTPBasicAuth(USERNAME, PASSWORD),
    headers={"Accept": "application/json"}
)

if response.status_code == 200:
    views_in_db = response.json()
    print(f"Views found in database: {len(views_in_db)}")
    for v in views_in_db:
        print(f"  ID: {v.get('id')} | Name: {v.get('name')}")
else:
    print(f"Failed with status {response.status_code}: {response.text[:200]}")

Views found in database: 2
  ID: 8ecff635-0597-447e-b59f-2b377a9cff0f | Name: collision_severity_summary
  ID: 1a8e1df5-c381-42fd-af08-2060a3e3ef94 | Name: collision_ml_features


In [7]:
if created_views.get("collision_ml_features"):
    view_id = created_views["collision_ml_features"]

    response = requests.get(
        f"{BASE_URL}/api/v1/database/{DATABASE_ID}/view/{view_id}/data",
        auth=HTTPBasicAuth(USERNAME, PASSWORD),
        headers={"Accept": "application/json"},
        params={"page": 0, "size": 5}
    )

    print(f"collision_ml_features spot check - Status: {response.status_code}")

    if response.status_code == 200:
        import pandas as pd
        data = response.json()
        df = pd.DataFrame(data)
        print(f"Rows returned: {len(df)}")
        print(f"Columns: {list(df.columns)}")
        print(df.head())
    else:
        print(f"Failed: {response.text[:200]}")
else:
    print("collision_ml_features was not created, skipping spot check")

collision_ml_features was not created, skipping spot check
